In [ ]:
from datasets import load_dataset
from tqdm import tqdm


In [ ]:
def analyze_ultrachat_contamination(sample_size=10000):
    print(f"Streaming {sample_size} prompts from UltraChat...")
    ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)

    # The keyword lists
    code_keywords = {
        "def ",
        "class ",
        "import ",
        "return ",
        "print(",
        "function",
        "python",
        "javascript",
        "html",
        "css",
        "c++",
        "java",
        "sql",
        "compile",
        "debug",
        "variable",
        "array",
        "syntax",
        "algorithm",
        "```",
    }
    math_keywords = {
        "calculate",
        "equation",
        "theorem",
        "integral",
        "derivative",
        "algebra",
        "geometry",
        "math",
        "\\frac",
        "\\sqrt",
        "sum of",
    }
    med_keywords = {
        "symptom",
        "diagnosis",
        "treatment",
        "patient",
        "syndrome",
        "mg ",
        "dosage",
        "clinical",
        "doctor",
        "disease",
        "infection",
    }

    stats = {
        "total_analyzed": 0,
        "contains_code": 0,
        "contains_math": 0,
        "contains_med": 0,
        "clean_general": 0,
    }

    for row in tqdm(ds, total=sample_size, desc="Analyzing"):
        if stats["total_analyzed"] >= sample_size:
            break

        messages = row.get("messages", [])
        if not messages or messages[0]["role"] != "user":
            continue

        prompt = messages[0]["content"].lower()
        stats["total_analyzed"] += 1

        # Check contamination
        has_code = any(kw in prompt for kw in code_keywords)
        has_math = any(kw in prompt for kw in math_keywords)
        has_med = any(kw in prompt for kw in med_keywords)

        if has_code:
            stats["contains_code"] += 1
        if has_math:
            stats["contains_math"] += 1
        if has_med:
            stats["contains_med"] += 1

        if not (has_code or has_math or has_med):
            stats["clean_general"] += 1

    # Print Report
    print("\n" + "=" * 50)
    print("ULTRACHAT CONTAMINATION REPORT")
    print("=" * 50)
    total = stats["total_analyzed"]
    print(f"Total Prompts Analyzed: {total:,}")
    print("-" * 50)

    code_pct = (stats["contains_code"] / total) * 100
    math_pct = (stats["contains_math"] / total) * 100
    med_pct = (stats["contains_med"] / total) * 100
    clean_pct = (stats["clean_general"] / total) * 100

    print(
        f"Contains CODE keywords:    {stats['contains_code']:>5,} ({code_pct:>5.1f}%)"
    )
    print(
        f"Contains MATH keywords:    {stats['contains_math']:>5,} ({math_pct:>5.1f}%)"
    )
    print(f"Contains MEDICAL keywords: {stats['contains_med']:>5,} ({med_pct:>5.1f}%)")
    print("-" * 50)
    print(
        f"CLEAN (No target keywords):{stats['clean_general']:>5,} ({clean_pct:>5.1f}%)"
    )
    print("=" * 50)

    # Note: A single prompt can contain multiple keyword types,
    # so percentages might add up to more than 100% when combined with Clean.


if __name__ == "__main__":
    analyze_ultrachat_contamination(100000)


# Analyze Squad/DailyDialog Contamination

In [ ]:
def analyze_dataset_contamination(dataset_name, sample_size):
    print(f"\nStreaming up to {sample_size} prompts from {dataset_name}...")
    ds = load_dataset(
        dataset_name, split="train", streaming=True, trust_remote_code=True
    )

    # The keyword lists
    code_keywords = {
        "def ",
        "class ",
        "import ",
        "return ",
        "print(",
        "function",
        "python",
        "javascript",
        "html",
        "css",
        "c++",
        "java",
        "sql",
        "compile",
        "debug",
        "variable",
        "array",
        "syntax",
        "algorithm",
        "```",
    }
    math_keywords = {
        "calculate",
        "equation",
        "theorem",
        "integral",
        "derivative",
        "algebra",
        "geometry",
        "math",
        "\\frac",
        "\\sqrt",
        "sum of",
    }
    med_keywords = {
        "symptom",
        "diagnosis",
        "treatment",
        "patient",
        "syndrome",
        "mg ",
        "dosage",
        "clinical",
        "doctor",
        "disease",
        "infection",
    }

    stats = {
        "total_analyzed": 0,
        "contains_code": 0,
        "contains_math": 0,
        "contains_med": 0,
        "clean_general": 0,
    }

    for row in tqdm(ds, total=sample_size, desc=f"Analyzing {dataset_name}"):
        if stats["total_analyzed"] >= sample_size:
            break

        # Handle different dataset structures
        prompt = ""
        if dataset_name == "squad_v2":
            prompt = row.get("question", "")
        elif dataset_name == "daily_dialog":
            dialog = row.get("dialog", [])
            if dialog:
                prompt = dialog[0]  # Grab the first utterance

        if not prompt:
            continue

        prompt = prompt.lower()
        stats["total_analyzed"] += 1

        # Check contamination
        has_code = any(kw in prompt for kw in code_keywords)
        has_math = any(kw in prompt for kw in math_keywords)
        has_med = any(kw in prompt for kw in med_keywords)

        if has_code:
            stats["contains_code"] += 1
        if has_math:
            stats["contains_math"] += 1
        if has_med:
            stats["contains_med"] += 1

        if not (has_code or has_math or has_med):
            stats["clean_general"] += 1

    # Print Report
    print("\n" + "=" * 50)
    print(f"{dataset_name.upper()} CONTAMINATION REPORT")
    print("=" * 50)
    total = stats["total_analyzed"]

    if total == 0:
        print("No prompts analyzed.")
        return

    print(f"Total Prompts Analyzed: {total:,}")
    print("-" * 50)

    code_pct = (stats["contains_code"] / total) * 100
    math_pct = (stats["contains_math"] / total) * 100
    med_pct = (stats["contains_med"] / total) * 100
    clean_pct = (stats["clean_general"] / total) * 100

    print(
        f"Contains CODE keywords:    {stats['contains_code']:>5,} ({code_pct:>5.1f}%)"
    )
    print(
        f"Contains MATH keywords:    {stats['contains_math']:>5,} ({math_pct:>5.1f}%)"
    )
    print(f"Contains MEDICAL keywords: {stats['contains_med']:>5,} ({med_pct:>5.1f}%)")
    print("-" * 50)
    print(
        f"CLEAN (No target keywords):{stats['clean_general']:>5,} ({clean_pct:>5.1f}%)"
    )
    print("=" * 50)


if __name__ == "__main__":
    # Test SQuAD (Max ~130k)
    analyze_dataset_contamination("squad_v2", 130000)
